In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import mlflow
import mlflow.sklearn

In [62]:
from pathlib import Path
import mlflow

# notebooks -> project root
project_root = Path.cwd().parent

mlflow.set_tracking_uri(f"file:///{project_root.as_posix()}/mlruns")
mlflow.set_experiment("Credit Card Fraud Detection")

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:///d:/Git_Repos/Risk-Aware-Credit-Decisioning-System/mlruns


In [63]:
df = pd.read_csv("../data/processed/fraud_processed.csv")

print(df.shape)
df.head()

(20000, 25)


,amount_usd,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,...,ip_country_mismatch,billing_shipping_mismatch,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,is_ai_generated_scam_attempt,merchant_risk_score,prior_disputes,is_fraud
0,42.86,Restaurants,Visa,OTP,Online,Android Phone,0,13.54,2,22.35,...,0,0,0,0.1,18,3,0,42.3,0,0
1,4.75,Online Retail,Mastercard,3D Secure,Online,Android Phone,0,0.71,2,35.28,...,0,0,0,25.8,12,4,0,28.3,0,0
2,77.18,Groceries,Mastercard,3D Secure,Online,Mac,0,0.35,5,9.82,...,0,1,0,42.3,5,0,0,24.7,1,0
3,1.69,Streaming,Visa,No Authentication,POS,Android Phone,0,3.42,6,19.72,...,0,0,0,28.9,22,6,0,56.2,1,0
4,261.68,Travel,Visa,3D Secure,In-App,iPhone,0,2.43,2,17.68,...,0,0,0,3.9,2,4,0,32.7,0,0


In [64]:
TARGET = "is_fraud"

X = df.drop(columns=[TARGET])
y = df[TARGET]

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [66]:
categorical_features = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
    "day_of_week"
]

boolean_features = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]

numerical_features = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "merchant_risk_score",
    "velocity_score",
    "cvv_retry_count",
    "prior_disputes"
]

print(categorical_features)
print(boolean_features)
print(numerical_features)

['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'day_of_week']
['is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'merchant_risk_score', 'velocity_score', 'cvv_retry_count', 'prior_disputes']


In [67]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [68]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [69]:
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:///d:/Git_Repos/Risk-Aware-Credit-Decisioning-System/mlruns


In [70]:
mlflow.end_run()
mlflow.set_experiment("Credit Card Fraud Detection")

with mlflow.start_run(run_name="logreg_pipeline_v1"):
    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Log parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Log metrics
    from sklearn.metrics import accuracy_score, roc_auc_score
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob))

    # Log model
    mlflow.sklearn.log_model(pipeline, artifact_path="model")

c:\Users\WIN10\miniconda3\envs\credit_fraud_detection\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/07/30 23:17:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [71]:

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.982
Precision: 0.4
Recall   : 0.11764705882352941
F1 Score : 0.18181818181818182
ROC AUC  : 0.9213654179881515
